# 1. Đọc dữ liệu và kiểm tra khả năng áp dụng IQR

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Tìm dataset gốc
DATA_PATH = None
SEARCH_ROOTS = [Path.cwd().resolve(), *Path.cwd().resolve().parents]

for folder in SEARCH_ROOTS:
    candidates = [
        folder / "data" / "raw" / "RT_IOT2022.csv",
        folder / "result" / "preprocessing" / "raw data" / "RT_IOT2022.csv",
        folder / "final_IoT" / "sample_project" / "result" / "preprocessing" / "raw data" / "RT_IOT2022.csv",
    ]
    for candidate in candidates:
        if candidate.is_file():
            DATA_PATH = candidate
            break
    if DATA_PATH is not None:
        break

if DATA_PATH is None:
    raise FileNotFoundError("Kh?ng t?m th?y RT_IOT2022.csv trong c?u tr?c project")

SAMPLE_ROOT = next(
    (
        candidate
        for folder in SEARCH_ROOTS
        for candidate in (folder, folder / "final_IoT" / "sample_project")
        if (candidate / "project").is_dir() and (candidate / "result").is_dir()
    ),
    None,
)
if SAMPLE_ROOT is None:
    raise FileNotFoundError("Kh?ng t?m th?y th? m?c g?c sample_project")

RUN_DIR = SAMPLE_ROOT / "result" / "eda" / "EDA_Thanhdai" / "03_Outliers_Correlation"
FIG_DIR = RUN_DIR / "images"
TABLE_DIR = RUN_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

# Lấy các thuộc tính số, bỏ cột chỉ số
num = df.select_dtypes(include=np.number).drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

# Tính các phân vị
q1 = num.quantile(0.25)
q3 = num.quantile(0.75)
iqr = q3 - q1

iqr_summary = pd.DataFrame({
    "Q1": q1,
    "Q3": q3,
    "IQR": iqr,
    "Zero (%)": num.eq(0).mean() * 100,
    "Unique Values": num.nunique()
})

iqr_summary["IQR = 0"] = iqr_summary["IQR"] == 0

print("=== KIỂM TRA IQR ===")
print("Tổng đặc trưng số:", num.shape[1])
print("Số cột IQR > 0:", (iqr > 0).sum())
print("Số cột IQR = 0:", (iqr == 0).sum())

print("\n=== CÁC CỘT IQR = 0 ===")
display(
    iqr_summary[
        iqr_summary["IQR = 0"]
    ].round(4)
)

iqr_summary.to_csv(
    TABLE_DIR / "03_iqr_summary.csv",
    encoding="utf-8-sig"
)

### Nhận xét: Khảo sát IQR

**Kết quả phân tích:**

- Dataset có 81 đặc trưng số khi không tính
  cột `Unnamed: 0`.
- Có 15 đặc trưng có IQR > 0.
- Có 66 đặc trưng có IQR = 0.
- Nhiều đặc trưng có Q1 và Q3 bằng nhau,
  cho thấy ít nhất 50% dữ liệu nằm tập trung
  tại cùng một giá trị hoặc khoảng rất hẹp.

**Vấn đề phát hiện:**

1. Phần lớn đặc trưng không có độ phân tán
   trong khoảng giữa Q1 và Q3.
2. Một số đặc trưng có nhiều giá trị 0
   hoặc giá trị lặp lại.
3. Khi IQR = 0, quy tắc phát hiện ngoại lệ
   bằng 1,5 × IQR có thể đánh dấu quá nhiều
   giá trị khác giá trị phổ biến.
4. Do đó, phương pháp IQR không phù hợp
   để áp dụng máy móc cho tất cả đặc trưng.

**Kết luận:**

Cần phân biệt các đặc trưng có IQR > 0
và IQR = 0 trước khi phân tích ngoại lệ.

Những đặc trưng có IQR = 0 không nhất thiết
là đặc trưng vô ích hoặc cần loại bỏ.

# 2. Phát hiện ngoại lệ bằng IQR

In [ ]:
valid_cols = iqr[iqr > 0].index

lower_bound = q1[valid_cols] - 1.5 * iqr[valid_cols]
upper_bound = q3[valid_cols] + 1.5 * iqr[valid_cols]

# Đánh dấu giá trị nằm ngoài khoảng IQR
outlier_mask = (
    num[valid_cols].lt(lower_bound) |
    num[valid_cols].gt(upper_bound)
)

outlier_count = outlier_mask.sum()

outlier_result = pd.DataFrame({
    "Q1": q1[valid_cols],
    "Q3": q3[valid_cols],
    "IQR": iqr[valid_cols],
    "Lower Bound": lower_bound,
    "Upper Bound": upper_bound,
    "Outlier Count": outlier_count,
    "Outlier (%)": outlier_count / len(df) * 100
})

outlier_result = outlier_result.sort_values(
    "Outlier Count",
    ascending=False
)

print("=== NGOẠI LỆ THEO IQR ===")
display(outlier_result.round(4))

print("\n=== TOP 10 CỘT CÓ NHIỀU GIÁ TRỊ NGOẠI LỆ ===")

display(
    outlier_result[
        ["Outlier Count", "Outlier (%)"]
    ].head(10).round(2)
)

# Vẽ số lượng ngoại lệ
top10 = outlier_result.head(10)

plt.figure(figsize=(11, 6))

sns.barplot(
    x=top10["Outlier Count"].values,
    y=top10.index,
    color="steelblue"
)

plt.title("Top 10 đặc trưng có nhiều ngoại lệ theo IQR")
plt.xlabel("Số lượng giá trị nằm ngoài ngưỡng IQR")
plt.ylabel("Đặc trưng")

plt.tight_layout()

plt.savefig(
    FIG_DIR / "03_iqr_outlier_counts.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

outlier_result.to_csv(
    TABLE_DIR / "03_iqr_outlier_results.csv",
    encoding="utf-8-sig"
)

### Nhận xét: Phát hiện ngoại lệ bằng IQR

**Kết quả phân tích:**

Trong 15 đặc trưng có IQR > 0:

- `flow_iat.avg`: 19.180 giá trị nằm ngoài
  ngưỡng IQR.
- `flow_iat.tot`: 19.180 giá trị.
- `flow_iat.max`: 19.180 giá trị.
- `flow_duration`: 19.171 giá trị.
- `active.max`: 19.165 giá trị.

**Vấn đề phát hiện:**

1. Một số đặc trưng có khoảng 15–16% số mẫu
   nằm ngoài ngưỡng IQR.
2. Giá trị trung tâm của nhiều đặc trưng
   tập trung trong khoảng rất hẹp.
3. Các giá trị nằm ngoài ngưỡng IQR
   chưa chắc là dữ liệu sai.
4. Trong bài toán phát hiện tấn công mạng,
   những giá trị bất thường có thể mang
   thông tin quan trọng về hành vi tấn công.

**Kết luận:**

Phát hiện nhiều giá trị bất thường theo
quy tắc IQR ở các đặc trưng liên quan
đến thời gian và hoạt động lưu lượng mạng.

# 3. Boxplot: trực quan hóa ngoại lệ

In [ ]:
selected_features = [
    "flow_duration",
    "flow_iat.tot",
    "active.max",
    "flow_pkts_payload.std"
]

fig, axes = plt.subplots(
    2, 2,
    figsize=(15, 9)
)

for ax, col in zip(axes.flat, selected_features):

    sns.boxplot(
        x=num[col],
        ax=ax,
        showfliers=True,
        flierprops={
            "marker": ".",
            "markersize": 1,
            "alpha": 0.3
        }
    )

    # Chỉ đổi thang trục để quan sát rõ hơn
    ax.set_xscale(
        "symlog",
        linthresh=max(float(iqr[col]), 1e-6)
    )

    ax.set_title(f"Boxplot - {col}")
    ax.set_xlabel(col)

plt.suptitle(
    "EDA 03 - Phân tích ngoại lệ bằng Boxplot",
    fontsize=14
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "03_boxplot_outliers.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

### Nhận xét: Boxplot

**Kết quả phân tích:**

- `flow_duration` có phần lớn giá trị tập trung
  trong khoảng rất nhỏ, nhưng vẫn xuất hiện
  những giá trị lớn hơn rất nhiều.
- `flow_iat.tot` có sự chênh lệch lớn giữa
  vùng dữ liệu trung tâm và các giá trị cực đại.
- `active.max` cũng xuất hiện nhiều giá trị
  nằm ngoài khoảng IQR.
- `flow_pkts_payload.std` cho thấy sự phân tán
  của các giá trị thống kê payload.

**Vấn đề phát hiện:**

1. Phân phối của một số đặc trưng có đuôi dài.
2. Có nhiều giá trị nằm ngoài ngưỡng IQR.
3. Khoảng biến thiên giữa các đặc trưng
   khác nhau đáng kể.
4. Những giá trị bất thường có thể phản ánh
   hành vi mạng khác với phần lớn dữ liệu.

**Kết luận:**

Boxplot xác nhận sự tồn tại của nhiều giá trị
bất thường theo quy tắc thống kê.

Tuy nhiên, đối với dữ liệu tấn công mạng,
không thể mặc định những giá trị này là nhiễu.

Cần tiếp tục kiểm tra mối quan hệ giữa
ngoại lệ và nhãn Attack_type trước khi
đưa ra quyết định xử lý.

# 4. Correlation Analysis: Pearson và Spearman

In [ ]:
# Loại cột hằng khỏi phép tính tương quan
# Không xóa cột khỏi dataset gốc
corr_data = num.loc[
    :, num.nunique(dropna=False) > 1
]

print("Số đặc trưng dùng tính tương quan:",
      corr_data.shape[1])

# Tính Pearson
pearson_corr = corr_data.corr(method="pearson")

# Tính Spearman
spearman_corr = corr_data.corr(method="spearman")

# Chỉ lấy mỗi cặp một lần, bỏ đường chéo
mask = np.triu(
    np.ones(pearson_corr.shape, dtype=bool),
    k=1
)

pearson_pairs = (
    pearson_corr
    .where(mask)
    .stack()
    .rename("Pearson")
)

spearman_pairs = (
    spearman_corr
    .where(mask)
    .stack()
    .rename("Spearman")
)

# Gộp kết quả
corr_pairs = pd.concat(
    [pearson_pairs, spearman_pairs],
    axis=1
).reset_index()

corr_pairs.columns = [
    "Feature 1",
    "Feature 2",
    "Pearson",
    "Spearman"
]

corr_pairs["Abs Pearson"] = (
    corr_pairs["Pearson"].abs()
)

corr_pairs["Abs Spearman"] = (
    corr_pairs["Spearman"].abs()
)

print("=== PHÂN TÍCH TƯƠNG QUAN ===")

print(
    "Số cặp |Pearson| >= 0.95:",
    (corr_pairs["Abs Pearson"] >= 0.95).sum()
)

print(
    "Số cặp |Spearman| >= 0.95:",
    (corr_pairs["Abs Spearman"] >= 0.95).sum()
)

print("\n=== TOP 20 CẶP PEARSON CAO NHẤT ===")

display(
    corr_pairs.sort_values(
        "Abs Pearson",
        ascending=False
    ).head(20).round(6)
)

# Lưu bảng kết quả
corr_pairs.to_csv(
    TABLE_DIR / "03_correlation_pairs.csv",
    index=False,
    encoding="utf-8-sig"
)

### Nhận xét: Pearson và Spearman Correlation
**Kết quả phân tích:**

Sau khi loại cột hằng khỏi phép tính tương quan:

- Phát hiện 52 cặp đặc trưng có
  |Pearson| >= 0.95.
- Phát hiện 180 cặp đặc trưng có
  |Spearman| >= 0.95.

Một số cặp có tương quan Pearson rất cao:

1. `flow_duration` và `flow_iat.tot`.
2. `bwd_pkts_per_sec` và `flow_pkts_per_sec`.
3. `fwd_pkts_per_sec` và `flow_pkts_per_sec`.
4. `bwd_bulk_bytes` và `bwd_bulk_packets`.
5. `flow_iat.max` và `idle.max`.

**Vấn đề phát hiện:**

1. Dataset tồn tại nhiều cặp đặc trưng có
   quan hệ tuyến tính hoặc đơn điệu rất mạnh.
2. Một số đặc trưng có khả năng chứa
   thông tin tương tự hoặc được tính toán
   từ những đại lượng liên quan.
3. Điều này làm xuất hiện nguy cơ dư thừa
   thông tin trong tập đặc trưng đầu vào.
4. Số cặp tương quan cao theo Spearman
   lớn hơn Pearson, cho thấy nhiều mối
   quan hệ đơn điệu không nhất thiết tuyến tính.

**Kết luận:**

Tập dữ liệu có dấu hiệu dư thừa đặc trưng.

Tuy nhiên, tương quan cao không đồng nghĩa
hai đặc trưng hoàn toàn giống nhau hoặc
một trong hai chắc chắn không có giá trị.

Cần xem xét thử Correlation Filter
trong giai đoạn thực nghiệm và đánh giá
ảnh hưởng đến hiệu quả MLP.

# 5. Correlation Heatmap và kiểm tra cặp đặc trưng tương quan cao

In [ ]:
# Chọn 10 cặp có Pearson tuyệt đối lớn nhất
top_pairs = corr_pairs.sort_values(
    "Abs Pearson",
    ascending=False
).head(10)

# Lấy danh sách cột xuất hiện trong 10 cặp
selected_corr_features = list(dict.fromkeys(
    top_pairs["Feature 1"].tolist()
    + top_pairs["Feature 2"].tolist()
))

print("Các đặc trưng được chọn:")
print(selected_corr_features)

fig, axes = plt.subplots(
    1, 2,
    figsize=(19, 8)
)

# Pearson Heatmap
sns.heatmap(
    pearson_corr.loc[
        selected_corr_features,
        selected_corr_features
    ],
    ax=axes[0],
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    annot=False
)

axes[0].set_title("Pearson Correlation Heatmap")

# Spearman Heatmap
sns.heatmap(
    spearman_corr.loc[
        selected_corr_features,
        selected_corr_features
    ],
    ax=axes[1],
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    annot=False
)

axes[1].set_title("Spearman Correlation Heatmap")

plt.tight_layout()

plt.savefig(
    FIG_DIR / "03_correlation_heatmap.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

# Kiểm tra chi tiết một cặp đặc trưng
feature_a = "flow_duration"
feature_b = "flow_iat.tot"

print("\n=== KIỂM TRA CẶP ĐẶC TRƯNG ===")

print("Feature 1:", feature_a)
print("Feature 2:", feature_b)

print(
    "Pearson:",
    pearson_corr.loc[feature_a, feature_b]
)

print(
    "Spearman:",
    spearman_corr.loc[feature_a, feature_b]
)

# Quan sát quan hệ bằng Scatterplot
sample = df[
    [feature_a, feature_b]
].sample(
    n=min(10000, len(df)),
    random_state=42
)

plt.figure(figsize=(9, 6))

plt.scatter(
    sample[feature_a],
    sample[feature_b],
    s=8,
    alpha=0.3
)

# Đổi thang hiển thị để dễ quan sát
plt.xscale("symlog", linthresh=1e-6)
plt.yscale("symlog", linthresh=1)

plt.title(
    "Quan hệ giữa flow_duration và flow_iat.tot"
)

plt.xlabel(feature_a)
plt.ylabel(feature_b)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "03_high_correlation_scatter.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

### Nhận xét: Correlation Heatmap
**Kết quả phân tích:**

- Heatmap cho thấy nhiều nhóm đặc trưng
  có hệ số tương quan gần 1.
- Cặp `flow_duration` và `flow_iat.tot`
  có tương quan tuyến tính gần như tuyệt đối.
- Một số đặc trưng liên quan đến số gói tin,
  tốc độ truyền và thời gian có quan hệ
  rất chặt chẽ.
- Scatterplot giúp quan sát trực tiếp mối
  liên hệ giữa các giá trị của hai đặc trưng.

**Vấn đề phát hiện:**

1. Một số đặc trưng có thể mang thông tin
   trùng lặp hoặc gần trùng lặp.
2. Tập đặc trưng có khả năng chứa nhiều
   thông tin dư thừa.
3. Việc sử dụng tất cả các đặc trưng
   có thể làm tăng độ phức tạp đầu vào.
4. Tương quan cao không chứng minh
   mô hình bị Data Leakage.

**Kết luận:**

Có cơ sở để thử nghiệm phương pháp
Correlation Filter nhằm giảm đặc trưng
dư thừa.

Tuy nhiên, cần đánh giá kết quả mô hình
trước và sau khi loại bỏ các đặc trưng.

Việc lựa chọn đặc trưng phải được thực hiện
trên tập Train trong giai đoạn huấn luyện,
không sử dụng Validation để quyết định
cột nào cần giữ hoặc loại bỏ.

# 6. Tổng hợp kết quả EDA 03 - Outliers & Correlation

### Những vấn đề phát hiện

| Vấn đề | Bằng chứng | Hướng nghiên cứu |
|---|---|---|
| Nhiều đặc trưng có IQR = 0 | 66/81 đặc trưng số | Xem xét phương pháp phát hiện ngoại lệ phù hợp |
| Giá trị nằm ngoài ngưỡng IQR | `flow_iat.tot`: 19.180 mẫu | Kiểm tra ngoại lệ theo từng lớp |
| Phân phối có đuôi dài | Quan sát Boxplot | Thử phương pháp biến đổi và chuẩn hóa phù hợp |
| Tương quan tuyến tính cao | 52 cặp có \|Pearson\| >= 0.95 | Thử Correlation Filter |
| Tương quan đơn điệu cao | 180 cặp có \|Spearman\| >= 0.95 | Kiểm tra quan hệ giữa các đặc trưng |
| Đặc trưng có khả năng dư thừa | `flow_duration` và `flow_iat.tot` | Đánh giá hiệu quả sau chọn đặc trưng |

### Đánh giá chung

Qua phân tích ngoại lệ và tương quan,
phát hiện hai vấn đề nổi bật.

**Thứ nhất: Đặc trưng có phân phối bất thường.**

Nhiều đặc trưng có IQR bằng 0, trong khi
một số đặc trưng khác có nhiều giá trị
nằm ngoài khoảng IQR.

Điều này cho thấy không nên áp dụng một
quy tắc xử lý ngoại lệ chung cho toàn bộ
dataset.

**Thứ hai: Đặc trưng có tương quan cao.**

Dataset xuất hiện nhiều cặp đặc trưng
có mối quan hệ chặt chẽ.

Đây là dấu hiệu cho thấy một số đặc trưng
có khả năng chứa thông tin dư thừa.

### Hướng nghiên cứu tiếp theo

- Phân tích ngoại lệ theo từng lớp Attack_type.
- Kiểm tra xem các giá trị bất thường có giúp
  nhận diện các loại tấn công hay không.
- Thử Correlation Filter trong các thực nghiệm.
- So sánh hiệu quả khi giữ và giảm đặc trưng.
- Không xóa ngoại lệ hoặc đặc trưng tương quan
  cao khi chưa có căn cứ đánh giá.

### Kết luận

EDA 03 đã phát hiện những vấn đề đáng chú ý
về ngoại lệ và tương quan giữa các đặc trưng.

Các phát hiện này sẽ được sử dụng làm cơ sở
đề xuất những phương pháp tiền xử lý phù hợp
cho mô hình MLP.

## Export the dated HTML run log

After running the analysis cells, save this notebook (Ctrl+S), then run the next cell. It copies the saved notebook with outputs into `result/eda/EDA_Thanhdai/<notebook>/` and exports a dated HTML file there.


In [ ]:
from datetime import datetime
import json
import shutil
import subprocess
import sys
from pathlib import Path

EXPORT_NAME = "03_Outliers_Correlation"
SEARCH_ROOTS = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
EXPORT_ROOT = next(
    (
        candidate
        for folder in SEARCH_ROOTS
        for candidate in (folder, folder / "final_IoT" / "sample_project")
        if (candidate / "project" / "eda").is_dir()
        and (candidate / "result" / "eda").is_dir()
    ),
    None,
)
if EXPORT_ROOT is None:
    raise FileNotFoundError("Could not locate sample_project from the current directory")

NOTEBOOK_PATH = EXPORT_ROOT / "project" / "eda" / f"{EXPORT_NAME}.ipynb"
RUN_DIR = EXPORT_ROOT / "result" / "eda" / "EDA_Thanhdai" / EXPORT_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

with NOTEBOOK_PATH.open(encoding="utf-8") as notebook_file:
    saved_notebook = json.load(notebook_file)
saved_code_cells = [
    cell for cell in saved_notebook.get("cells", [])
    if cell.get("cell_type") == "code"
]
if not any(cell.get("outputs") for cell in saved_code_cells):
    raise RuntimeError(
        "No saved execution outputs found. Run the analysis, save the notebook (Ctrl+S), then rerun this export cell."
    )

RUN_NOTEBOOK = RUN_DIR / NOTEBOOK_PATH.name
shutil.copy2(NOTEBOOK_PATH, RUN_NOTEBOOK)
HTML_STEM = f"{EXPORT_NAME}_{datetime.now():%Y%m%d}"
subprocess.run(
    [
        sys.executable,
        "-m",
        "nbconvert",
        "--to",
        "html",
        "--output-dir",
        str(RUN_DIR),
        "--output",
        HTML_STEM,
        str(RUN_NOTEBOOK),
    ],
    check=True,
)
print(f"HTML export: {RUN_DIR / (HTML_STEM + '.html')}")
